# Análise Epidemiológica e Distribuição de Vacinas em Destaque

**Projeto:** PyAnalytics — UFSC Araranguá

**Autoras:** Beatriz Goulart e Esther de Oliveira  

**Arquivo de Origem:** `vacinas_analitico.csv`  

---

## Contexto e Objetivos

A análise dos registros de vacinação permite identificar quais imunizantes apresentam maior ou menor participação na base e compreender como essa distribuição se comporta de acordo com diferentes dimensões, como período, estratégia, dose e território.

Esta análise busca destacar os principais imunizantes presentes na base e organizar sua distribuição de forma que facilite a interpretação dos registros de vacinação, permitindo:

1. Identificar quais vacinas apresentam maior volume de aplicações registradas.
2. Comparar a participação dos imunizantes por diferentes medidas e recortes.
3. Verificar se a posição das vacinas se modifica conforme o período e a estratégia.
4. Avaliar se os registros estão concentrados em poucos imunizantes ou distribuídos de forma mais equilibrada.
5. Preservar diferenças entre apresentações, estratégias e doses para evitar agrupamentos inadequados.

## 1. Bibliotecas e configuração

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_colwidth", None)


## 2. Carregamento da base

In [ ]:
from pathlib import Path

caminhos = [
    Path("../../data/processed/vacinas_analitico.csv"),
    Path("../data/processed/vacinas_analitico.csv"),
    Path("data/processed/vacinas_analitico.csv"),
    Path("vacinas_analitico.csv")
]

file_path = next((caminho for caminho in caminhos if caminho.exists()), None)

if file_path is None:
    raise FileNotFoundError("O arquivo não foi encontrado.")

df = pd.read_csv(file_path)

print(f"Arquivo utilizado: {file_path}")
display(df.head())


## 3. Estrutura da base

In [ ]:
print(f"Registros: {df.shape[0]:,}".replace(",", "."))
print(f"Colunas: {df.shape[1]}")

display(pd.DataFrame({
    "Coluna": df.columns,
    "Tipo": df.dtypes.astype(str).values,
    "Valores ausentes": df.isna().sum().values,
    "Valores distintos": [df[col].nunique(dropna=True) for col in df.columns]
}))

display(df.head())


## 4. Qualidade dos dados

In [ ]:
qualidade = pd.DataFrame({
    "Indicador": [
        "Registros",
        "Colunas",
        "Registros duplicados",
        "Células ausentes"
    ],
    "Valor": [
        len(df),
        df.shape[1],
        int(df.duplicated().sum()),
        int(df.isna().sum().sum())
    ]
})

display(qualidade)

ausentes = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .rename_axis("Coluna")
    .reset_index(name="Valores ausentes")
)

display(ausentes.head(15))


## 5. Tratamento da data

In [ ]:
df["data_da_aplicacao"] = pd.to_datetime(
    df["data_da_aplicacao"],
    dayfirst=True,
    errors="coerce"
)

df["ano"] = df["data_da_aplicacao"].dt.year
df["mes"] = df["data_da_aplicacao"].dt.to_period("M").astype("string")

print("Período inicial:", df["data_da_aplicacao"].min())
print("Período final:", df["data_da_aplicacao"].max())
print("Datas inválidas:", int(df["data_da_aplicacao"].isna().sum()))

display(
    df[["data_da_aplicacao", "ano", "mes"]]
    .head()
)


In [ ]:
df["estrategia"] = df["estrategia"].astype("string").str.strip()
df["estrategia"] = df["estrategia"].replace({
    "Rotina (1)": "Rotina",
    "Pós": "Pós-exposição",
    "Monitoramento das Estratégias de Vacinação": "MEV"
})

df["dose"] = df["dose"].astype("string").str.strip()
df["dose"] = df["dose"].replace({
    "PROFILAXIA/TRATAMENTO COM 1 FRASCO-AMPOLA/AMPOLA": "PROFILAXIA/TRATAMENTO"
})

def rotulo_vacina(nome):
    if pd.isna(nome):
        return "Não informado"
    nome = re.sub(r"^Vacina\s+", "", str(nome).strip(), flags=re.IGNORECASE)
    nome = re.sub(r"\s+", " ", nome)
    nome = nome.replace("COVID-19-RNAm, ", "COVID-19 ")
    nome = nome.replace("COVID-19-RNAm,", "COVID-19")
    return nome.strip()

def rotulo_estrategia(nome):
    if pd.isna(nome):
        return "Não informado"
    return {
        "Rotina (1)": "Rotina",
        "Pós": "Pós-exposição",
        "Monitoramento das Estratégias de Vacinação": "MEV"
    }.get(str(nome).strip(), str(nome).strip())

print("Categorias normalizadas.")
print("Legenda: MEV = Monitoramento das Estratégias de Vacinação.")

In [ ]:
vacinas = (
    df["nome_da_vacina"]
    .value_counts(dropna=False)
    .rename_axis("Vacina_original")
    .reset_index(name="Aplicações")
)
vacinas["Imunizante"] = vacinas["Vacina_original"].map(rotulo_vacina)
vacinas["Percentual (%)"] = vacinas["Aplicações"] / len(df) * 100
display(vacinas[["Imunizante", "Aplicações", "Percentual (%)"]].head(15))

## 7. Ranking das vacinas

In [ ]:
top_vacinas = vacinas.head(10).sort_values("Aplicações")

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    top_vacinas["Imunizante"].astype(str),
    top_vacinas["Aplicações"]
)

ax.set_title("Imunizantes mais aplicados", fontsize=17, fontweight="bold", pad=15)
ax.set_xlabel("Número de aplicações")
ax.set_ylabel("")

for bar, valor in zip(bars, top_vacinas["Aplicações"]):
    ax.text(
        bar.get_width() + top_vacinas["Aplicações"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{valor:,}".replace(",", "."),
        va="center",
        fontsize=10,
        fontweight="bold"
    )

ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()


## 8. Participação percentual dos imunizantes

In [ ]:
top_percentual = vacinas.head(10).sort_values("Percentual (%)")

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    top_percentual["Imunizante"].astype(str),
    top_percentual["Percentual (%)"]
)

ax.set_title("Participação dos imunizantes", fontsize=17, fontweight="bold", pad=15)
ax.set_xlabel("Participação (%)")
ax.set_ylabel("")

for bar, valor in zip(bars, top_percentual["Percentual (%)"]):
    ax.text(
        bar.get_width() + 0.15,
        bar.get_y() + bar.get_height() / 2,
        f"{valor:.1f}%",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()


## 9. Concentração das aplicações

In [ ]:
pareto = vacinas.copy()
pareto["Acumulado (%)"] = pareto["Percentual (%)"].cumsum()

x = np.arange(len(pareto))
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.bar(
    x,
    pareto["Percentual (%)"],
    alpha=0.85
)

ax1.set_xlabel("Imunizantes em ordem de frequência")
ax1.set_ylabel("Participação individual (%)")
ax1.set_title(
    "Concentração das aplicações",
    fontsize=17,
    fontweight="bold",
    pad=15
)

ax2 = ax1.twinx()

ax2.plot(
    x,
    pareto["Acumulado (%)"],
    marker="o",
    linewidth=2.5,
    markersize=4
)

ax2.set_ylabel("Participação acumulada (%)")
ax2.set_ylim(0, 105)

n = len(pareto)
passo = max(1, n // 10)
indices = list(range(0, n, passo))

ax1.set_xticks(indices)
ax1.set_xticklabels(
    [str(pareto.iloc[i]["Imunizante"])[:18] for i in indices],
    rotation=45,
    ha="right"
)

ax1.grid(axis="y", alpha=0.2)
ax1.spines["top"].set_visible(False)
ax2.spines["top"].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Os 5 imunizantes mais frequentes representam {pareto.head(5)['Percentual (%)'].sum():.1f}% dos registros.")
print(f"Os 10 imunizantes mais frequentes representam {pareto.head(10)['Percentual (%)'].sum():.1f}% dos registros.")


## 10. Distribuição por estratégia

In [ ]:
estrategias = (
    df["estrategia"]
    .value_counts(dropna=False)
    .rename_axis("Estratégia")
    .reset_index(name="Aplicações")
)

estrategias["Percentual (%)"] = estrategias["Aplicações"] / len(df) * 100

display(estrategias)

fig, ax = plt.subplots(figsize=(10, 5.5))

bars = ax.bar(
    estrategias["Estratégia"].astype(str),
    estrategias["Aplicações"]
)

ax.set_title("Aplicações por estratégia", fontsize=17, fontweight="bold", pad=15)
ax.set_ylabel("Aplicações")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")

for bar, valor in zip(bars, estrategias["Aplicações"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{valor:,}".replace(",", "."),
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
tabela_vacina_estrategia = pd.crosstab(
    df["nome_da_vacina"],
    df["estrategia"]
)

display(tabela_vacina_estrategia.head(15))


In [ ]:
heatmap = pd.crosstab(df["nome_da_vacina"], df["estrategia"], dropna=False)
top_nomes = vacinas.head(12)["Vacina_original"].tolist()
heatmap = heatmap.reindex(index=[v for v in top_nomes if v in heatmap.index], fill_value=0)
heatmap = heatmap.loc[heatmap.sum(axis=1) > 0, heatmap.sum(axis=0) > 0]
heatmap.index = [rotulo_vacina(v) for v in heatmap.index]
heatmap.columns = [rotulo_estrategia(v) for v in heatmap.columns]

if heatmap.empty:
    print("Não há dados suficientes para gerar o mapa de calor.")
else:
    fig, ax = plt.subplots(figsize=(12, max(5.5, 0.45 * len(heatmap))))
    sns.heatmap(
        heatmap,
        annot=True,
        fmt=".0f",
        cmap="Blues",
        linewidths=0.6,
        linecolor="white",
        cbar_kws={"label": "Aplicações"},
        ax=ax
    )
    ax.set_title("Mapa de calor: imunizantes por estratégia", fontsize=16, fontweight="bold", pad=15)
    ax.set_xlabel("Estratégia")
    ax.set_ylabel("Imunizante")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
principais = vacinas.head(5)["Vacina_original"].tolist()

mensal = (
    df[df["nome_da_vacina"].isin(principais)]
    .groupby(["mes", "nome_da_vacina"])
    .size()
    .unstack(fill_value=0)
)

display(mensal)


In [ ]:
if mensal.empty:
    print("Não há registros suficientes para gerar a evolução mensal.")
else:
    ax = mensal.plot(
        figsize=(12, 6),
        marker="o",
        linewidth=2.5,
        markersize=6
    )

    ax.set_title(
        "Evolução das aplicações dos principais imunizantes",
        fontsize=17,
        fontweight="bold",
        pad=15
    )
    ax.set_xlabel("Mês")
    ax.set_ylabel("Aplicações")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.2)

    plt.xticks(rotation=45, ha="right")
    plt.legend(
        title="Imunizante",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False
    )

    plt.tight_layout()
    plt.show()


## 15. Mudança de posição das vacinas ao longo do tempo

In [ ]:
ranking_mensal = (
    df.dropna(subset=["mes"])
    .groupby(["mes", "nome_da_vacina"])
    .size()
    .reset_index(name="Aplicações")
)
ranks = ranking_mensal.groupby("mes")["Aplicações"].rank(method="min", ascending=False)
ranking_mensal["Posição"] = ranks.astype(int)
ranking_mensal["Vacina"] = ranking_mensal["nome_da_vacina"].map(rotulo_vacina)

display(ranking_mensal.sort_values(["mes", "Posição"]).head(30))

variacao_posicao = (
    ranking_mensal.groupby("nome_da_vacina")["Posição"]
    .agg(Posição_mínima="min", Posição_máxima="max", Posições_diferentes="nunique")
    .sort_values(["Posições_diferentes", "Posição_mínima"], ascending=[False, True])
)
display(variacao_posicao.head(15))

variacao_grafico = variacao_posicao.head(10).reset_index()
variacao_grafico["Imunizante"] = variacao_grafico["nome_da_vacina"].map(rotulo_vacina)
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(variacao_grafico["Imunizante"].iloc[::-1], variacao_grafico["Posições_diferentes"].iloc[::-1])
ax.set_title("Variação da posição dos imunizantes ao longo dos meses", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Quantidade de posições diferentes")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()


## 16. Distribuição anual

In [ ]:
anual = (
    df.dropna(subset=["ano"])
    .groupby(["ano", "nome_da_vacina"])
    .size()
    .reset_index(name="Aplicações")
)

anual_top5 = anual[anual["nome_da_vacina"].isin(principais)].copy()
anual_top5["Imunizante"] = anual_top5["nome_da_vacina"].map(rotulo_vacina)

display(anual_top5)

if anual_top5.empty:
    print("Não há dados anuais suficientes para gerar o gráfico.")
else:
    fig, ax = plt.subplots(figsize=(12, 6))

    sns.barplot(
        data=anual_top5,
        x="ano",
        y="Aplicações",
        hue="Imunizante",
        ax=ax
    )

    ax.set_title(
        "Aplicações anuais dos 5 principais imunizantes",
        fontsize=16,
        fontweight="bold",
        pad=15
    )
    ax.set_xlabel("Ano")
    ax.set_ylabel("Aplicações registradas")

    plt.tight_layout()
    plt.show()


## 17. Distribuição territorial por bairro

In [ ]:
colunas_bairro = [
    coluna for coluna in df.columns
    if "bairro" in coluna.lower()
]

print("Colunas relacionadas a bairro:", colunas_bairro)

if not colunas_bairro:
    bairros = pd.DataFrame(columns=["Bairro", "Aplicações", "Percentual (%)"])
    print("A base não possui uma coluna de bairro.")
else:
    coluna_bairro = colunas_bairro[0]

    bairros = (
        df[coluna_bairro]
        .value_counts(dropna=False)
        .rename_axis("Bairro")
        .reset_index(name="Aplicações")
    )

    bairros["Percentual (%)"] = bairros["Aplicações"] / len(df) * 100

    display(bairros.head(15))


## 18. Principais bairros

In [ ]:
if bairros.empty:
    print("Não há dados de bairro para gerar o gráfico.")
else:
    top_bairros = bairros.head(10).sort_values("Aplicações")

    fig, ax = plt.subplots(figsize=(11, 6))

    bars = ax.barh(
        top_bairros["Bairro"].astype(str),
        top_bairros["Aplicações"]
    )

    ax.set_title(
        "Bairros com mais aplicações",
        fontsize=17,
        fontweight="bold",
        pad=15
    )
    ax.set_xlabel("Número de aplicações")
    ax.set_ylabel("")

    for bar, valor in zip(bars, top_bairros["Aplicações"]):
        ax.text(
            bar.get_width() + top_bairros["Aplicações"].max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{valor:,}".replace(",", "."),
            va="center",
            fontsize=10,
            fontweight="bold"
        )

    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25)

    plt.tight_layout()
    plt.show()


## 19. Distribuição por dose

In [ ]:
doses = (
    df["dose"]
    .value_counts(dropna=False)
    .rename_axis("Dose")
    .reset_index(name="Aplicações")
)

doses["Percentual (%)"] = doses["Aplicações"] / len(df) * 100

display(doses.head(15))


## 20. Principais doses

In [ ]:
top_doses = doses.head(10).sort_values("Aplicações")

fig, ax = plt.subplots(figsize=(11, 6.5))

bars = ax.barh(
    top_doses["Dose"].astype(str),
    top_doses["Aplicações"]
)

ax.set_title(
    "Principais doses",
    fontsize=17,
    fontweight="bold",
    pad=15
)
ax.set_xlabel("Número de aplicações")
ax.set_ylabel("")

for bar, valor in zip(bars, top_doses["Aplicações"]):
    ax.text(
        bar.get_width() + top_doses["Aplicações"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{valor:,}".replace(",", "."),
        va="center",
        fontsize=10,
        fontweight="bold"
    )

ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()


## 21. Combinação entre vacina, estratégia e dose

In [ ]:
combinacoes = (
    df.groupby(
        ["nome_da_vacina", "estrategia", "dose"],
        dropna=False
    )
    .size()
    .reset_index(name="Aplicações")
    .sort_values("Aplicações", ascending=False)
)

combinacoes["Percentual (%)"] = (
    combinacoes["Aplicações"] / len(df) * 100
)

display(combinacoes.head(20))


## 22. Vacina, estratégia e dose ao longo do tempo

In [ ]:
combinacoes_mensais = (
    df.dropna(subset=["mes"])
    .groupby(["mes", "nome_da_vacina", "estrategia", "dose"], dropna=False)
    .size().reset_index(name="Aplicações")
)
chaves_top = (
    combinacoes_mensais.groupby(["nome_da_vacina", "estrategia", "dose"])["Aplicações"]
    .sum().sort_values(ascending=False).head(5).index
)
top_mensais = combinacoes_mensais[combinacoes_mensais.set_index(["nome_da_vacina", "estrategia", "dose"]).index.isin(chaves_top)].copy()
top_mensais["Combinação"] = (top_mensais["nome_da_vacina"].map(rotulo_vacina) + " | " + top_mensais["estrategia"].fillna("Não informado") + " | " + top_mensais["dose"].fillna("Não informado"))
display(top_mensais.sort_values(["mes", "Aplicações"], ascending=[True, False]).head(30))

serie_combinacoes = top_mensais.pivot_table(index="mes", columns="Combinação", values="Aplicações", aggfunc="sum", fill_value=0)
if not serie_combinacoes.empty:
    ax = serie_combinacoes.plot(figsize=(13, 6.5), marker="o", linewidth=2)
    ax.set_title("Evolução mensal das principais combinações", fontsize=16, fontweight="bold", pad=15)
    ax.set_xlabel("Mês")
    ax.set_ylabel("Aplicações")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.2)
    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Combinação", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout()
    plt.show()


## 23. Conferência de códigos e nomes das vacinas

In [ ]:
if "codigo_da_vacina" not in df.columns:
    print("A coluna codigo_da_vacina não está disponível na base.")
else:
    codigos_vacinas = (
        df[["codigo_da_vacina", "nome_da_vacina"]]
        .drop_duplicates()
        .sort_values(["codigo_da_vacina", "nome_da_vacina"])
    )

    display(codigos_vacinas)

    conflitos = (
        codigos_vacinas
        .groupby("codigo_da_vacina")["nome_da_vacina"]
        .nunique()
        .loc[lambda serie: serie > 1]
    )

    print("Códigos associados a mais de um nome:", len(conflitos))

    if len(conflitos) > 0:
        display(
            codigos_vacinas[
                codigos_vacinas["codigo_da_vacina"].isin(conflitos.index)
            ]
        )


## 24. Preservação de apresentações próximas

In [ ]:
covid_mask = df["nome_da_vacina"].astype("string").str.contains("COVID", case=False, na=False)
covid = (
    df.loc[covid_mask, "nome_da_vacina"]
    .value_counts(dropna=False)
    .rename_axis("Vacina_original")
    .reset_index(name="Aplicações")
)
covid["Imunizante"] = covid["Vacina_original"].map(rotulo_vacina)

display(covid[["Imunizante", "Aplicações"]])

if not covid.empty:
    dados = covid.sort_values("Aplicações")
    fig, ax = plt.subplots(figsize=(12, max(5, 0.55 * len(dados))))
    bars = ax.barh(dados["Imunizante"], dados["Aplicações"])
    ax.set_title("Categorias de COVID-19", fontsize=16, fontweight="bold", pad=15)
    ax.set_xlabel("Aplicações")
    ax.set_ylabel("")
    limite = max(dados["Aplicações"].max(), 1)
    for bar, valor in zip(bars, dados["Aplicações"]):
        ax.text(bar.get_width() + limite * 0.01, bar.get_y() + bar.get_height()/2, f"{valor:,}".replace(",", "."), va="center", fontsize=9)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

nirsevimabe = (
    df.loc[df["nome_da_vacina"].astype("string").str.contains("NIRSEVIMABE", case=False, na=False), "nome_da_vacina"]
    .value_counts(dropna=False)
    .rename_axis("Vacina_original")
    .reset_index(name="Aplicações")
)
nirsevimabe["Imunizante"] = nirsevimabe["Vacina_original"].map(rotulo_vacina)
display(nirsevimabe[["Imunizante", "Aplicações"]])

if "vacina_estrategia_e_dose" in df.columns:
    apresentacoes = df["vacina_estrategia_e_dose"].value_counts(dropna=False).rename_axis("Imunizante + estratégia + dose").reset_index(name="Aplicações")
    apresentacoes["Percentual (%)"] = apresentacoes["Aplicações"] / len(df) * 100
    display(apresentacoes.head(20))
else:
    apresentacoes = pd.DataFrame()
    print("A coluna vacina_estrategia_e_dose não está disponível na base.")

## 25. Comparação entre imunizantes por diferentes recortes

In [ ]:
top12 = vacinas.head(12)["Vacina_original"].tolist()
recorte_estrategia = (
    df[df["nome_da_vacina"].isin(top12)]
    .groupby(["nome_da_vacina", "estrategia"], dropna=False)
    .size().reset_index(name="Aplicações")
)
recorte_estrategia["Percentual dentro do imunizante (%)"] = (
    recorte_estrategia["Aplicações"] /
    recorte_estrategia.groupby("nome_da_vacina")["Aplicações"].transform("sum") * 100
)
recorte_estrategia["Imunizante"] = recorte_estrategia["nome_da_vacina"].map(rotulo_vacina)
recorte_estrategia["Estratégia"] = recorte_estrategia["estrategia"].map(rotulo_estrategia)

display(recorte_estrategia[["Imunizante", "Estratégia", "Aplicações", "Percentual dentro do imunizante (%)"]].sort_values(["Imunizante", "Aplicações"], ascending=[True, False]))

grafico_estrategia = recorte_estrategia.pivot_table(index="Imunizante", columns="Estratégia", values="Percentual dentro do imunizante (%)", aggfunc="sum", fill_value=0)
fig, ax = plt.subplots(figsize=(12, 7))
grafico_estrategia.plot(kind="barh", stacked=True, ax=ax, width=0.78)
ax.set_title("Estratégias dentro dos principais imunizantes", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Participação dentro do imunizante (%)")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.grid(axis="x", alpha=0.2)
ax.legend(title="Estratégia", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

## 26. Aplicações e pessoas

In [ ]:
possiveis_identificadores = [
    coluna for coluna in df.columns
    if any(
        termo in coluna.lower()
        for termo in ["cpf", "cns", "id_pessoa", "id_paciente", "paciente", "pessoa"]
    )
]

total_aplicacoes = len(df)

print("Possíveis identificadores de pessoa:", possiveis_identificadores)
print(f"Total de aplicações registradas: {total_aplicacoes:,}".replace(",", "."))

# Um gráfico com uma única barra não acrescenta informação.
# Quando não existe identificador de pessoa, apresentamos o total como indicador visual.
if possiveis_identificadores:
    identificador = possiveis_identificadores[0]
    pessoas_distintas = df[identificador].nunique(dropna=True)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    indicadores = [
        ("Aplicações registradas", total_aplicacoes),
        ("Pessoas distintas", pessoas_distintas),
    ]

    for ax, (titulo, valor) in zip(axes, indicadores):
        ax.text(
            0.5, 0.55,
            f"{valor:,}".replace(",", "."),
            ha="center",
            va="center",
            fontsize=28,
            fontweight="bold"
        )
        ax.text(
            0.5, 0.25,
            titulo,
            ha="center",
            va="center",
            fontsize=12
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")

    fig.suptitle(
        "Resumo da população registrada",
        fontsize=16,
        fontweight="bold",
        y=0.98
    )
    plt.tight_layout()
    plt.show()

else:
    print(
        "Não foi identificado um campo de pessoa. "
        "O total abaixo representa aplicações registradas, não pessoas únicas."
    )

    fig, ax = plt.subplots(figsize=(10, 3.8))

    ax.text(
        0.5, 0.58,
        f"{total_aplicacoes:,}".replace(",", "."),
        ha="center",
        va="center",
        fontsize=36,
        fontweight="bold"
    )
    ax.text(
        0.5, 0.28,
        "aplicações registradas",
        ha="center",
        va="center",
        fontsize=14
    )

    ax.set_title(
        "Total de registros analisados",
        fontsize=16,
        fontweight="bold",
        pad=18
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    plt.tight_layout()
    plt.show()


## 27. Síntese dos resultados

In [ ]:
total_aplicacoes = len(df)
quantidade_imunizantes = df["nome_da_vacina"].nunique()
quantidade_estrategias = df["estrategia"].nunique()
imunizante_principal = vacinas.iloc[0]["Imunizante"]
participacao_principal = vacinas.iloc[0]["Percentual (%)"]
participacao_top5 = vacinas.head(5)["Percentual (%)"].sum()
participacao_top10 = vacinas.head(10)["Percentual (%)"].sum()

print(f"Total de aplicações analisadas: {total_aplicacoes:,}".replace(",", "."))
print(f"Quantidade de imunizantes: {quantidade_imunizantes}")
print(f"Quantidade de estratégias: {quantidade_estrategias}")
print(f"Imunizante mais frequente: {imunizante_principal}")
print(f"Participação do imunizante mais frequente: {participacao_principal:.1f}%")
print(f"Participação acumulada dos 5 principais imunizantes: {participacao_top5:.1f}%")
print(f"Participação acumulada dos 10 principais imunizantes: {participacao_top10:.1f}%")

if not bairros.empty:
    print(f"Quantidade de bairros: {df[coluna_bairro].nunique(dropna=True)}")
else:
    print("Quantidade de bairros: não disponível")

print(f"Período analisado: {df['data_da_aplicacao'].min()} até {df['data_da_aplicacao'].max()}")

resumo_grafico = vacinas.head(10).sort_values("Aplicações")
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(resumo_grafico["Imunizante"], resumo_grafico["Aplicações"])
ax.set_title("Resumo: 10 imunizantes com mais aplicações", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Aplicações")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()


## 28. Respostas às perguntas orientadoras

In [ ]:
print("1. Quais vacinas mais se destacam e por quais medidas?")
display(vacinas.head(10)[["Imunizante", "Aplicações", "Percentual (%)"]])

print("2. A posição das vacinas muda conforme o período?")
display(variacao_posicao.head(10))

print("3. A base se concentra em poucos imunizantes ou se distribui de forma mais equilibrada?")
print(f"Os 5 primeiros imunizantes representam {participacao_top5:.1f}% dos registros.")
print(f"Os 10 primeiros imunizantes representam {participacao_top10:.1f}% dos registros.")

print("4. As apresentações próximas permanecem diferenciadas?")
if "vacina_estrategia_e_dose" in df.columns:
    print(f"Foram identificadas {len(apresentacoes)} combinações distintas de vacina, estratégia e dose.")
else:
    print("A coluna específica de combinação não está disponível; a diferenciação foi analisada pelas colunas individuais.")

print("5. Como estratégia e dose modificam a leitura da distribuição?")
display(recorte_estrategia.sort_values("Aplicações", ascending=False).head(20))

if not variacao_posicao.empty:
    dados = variacao_posicao.head(10).reset_index()
    dados["Imunizante"] = dados["nome_da_vacina"].map(rotulo_vacina)
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(dados["Imunizante"].iloc[::-1], dados["Posições_diferentes"].iloc[::-1])
    ax.set_title("Variação de posição dos imunizantes ao longo do período", fontsize=16, fontweight="bold", pad=15)
    ax.set_xlabel("Posições diferentes observadas")
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
    ax.spines[["top", "right", "left"]].set_visible(False)
    plt.tight_layout()
    plt.show()


## 29. Limitações

- Os registros representam aplicações, não necessariamente pessoas únicas.
- A frequência de aplicações não representa, sozinha, cobertura vacinal.
- Os resultados dependem do período e da qualidade dos dados disponíveis.
- A análise descreve a distribuição dos registros e não mede eficácia clínica dos imunizantes.
- Bairro e sala de vacinação não são considerados equivalentes.
- Quando uma variável específica não está presente na base, a análise correspondente não é inventada ou substituída por outra variável.

## Conclusão

A análise exploratória permite identificar quais imunizantes possuem maior presença na base, comparar sua participação por diferentes medidas e observar como os registros se distribuem segundo período, estratégia, dose e território.

O ranking mostra os imunizantes com maior volume de aplicações, enquanto a participação percentual e a curva acumulada permitem avaliar a concentração dos registros. A análise temporal verifica mudanças na posição relativa das vacinas, e os cruzamentos com estratégia e dose preservam diferenças importantes entre os registros.

As apresentações próximas também são mantidas separadas quando a base fornece essa informação, evitando agrupamentos inadequados. Dessa forma, os resultados permanecem baseados nas dimensões efetivamente disponíveis no arquivo `vacinas_analitico.csv`.